# {{ cookiecutter.project_name }}
Device data for the launched patient, from JupyterHealth Exchange.

In [ ]:
# [SCAFFOLDED] launch context + data fetch — usually untouched
from provider_app import launch_context, jhe_auth, jhe_data, patient_resolver

ctx = launch_context.current()
client = jhe_auth.client_for_launch(ctx)

# If the launching provider's JHE account isn't authorized to see this patient
# (JupyterHealth RBAC), patient_resolver raises PatientNotInJHE — catch it so the
# dashboard shows a clean message instead of a traceback. access_note is None on success.
access_note = None
data = {}
try:
    data = jhe_data.fetch(ctx.patient_mrn, client=client, types=["heart_rate", "sleep", "steps"])  # <-- edit to the data types you want
except patient_resolver.PatientNotInJHE:
    access_note = (
        f"You don't have access to this patient's data in JupyterHealth "
        f"(MRN {ctx.patient_mrn}). They may not be enrolled, or your JupyterHealth "
        f"account isn't authorized to view them."
    )

In [ ]:
# ──────── ADD YOUR ANALYTICS + VISUALIZATION BELOW ────────
# Example: plot the first requested data type. Replace this with your own analytics.
if access_note:
    print("🔒 " + access_note)
elif not data:
    print("No data returned for this patient.")
else:
    import plotly.express as px

    data_type, df = next(iter(data.items()))
    if df.empty:
        print(f"No {data_type} data for this patient.")
    else:
        value_cols = [c for c in df.columns if c.endswith("_value")]
        y = value_cols[0] if value_cols else df.columns[-1]
        fig = px.line(df, x="effective_time_frame_date_time", y=y, title=data_type)
        fig.show()